# Unit 4 Assignment — Self-Evaluating Agentic RAG System
## Space Exploration Knowledge Base

**Student**: Your Name  
**Topic chosen**: Space Exploration & Planetary Science  
**Why**: Well-defined facts, 10+ distinct verifiable claims, ideal for RAG faithfulness testing.

---

### System Architecture

```
USER QUESTION
      │
      ▼
┌─────────────────────────────────┐
│  AGENT 1: RAG Retriever         │
│  FAISS + llama-3.3-70b          │
│  → answer + retrieved context   │
└────────────────┬────────────────┘
                 │
                 ▼
┌─────────────────────────────────┐
│  AGENT 2: Quality Evaluator     │
│  DeepEval (Faithfulness +       │
│  AnswerRelevancy, threshold=0.7)│
│  → PASS / FAIL + scores         │
└────────┬────────────────────────┘
         │
    ┌────┴─────┐
  PASS      FAIL
    │          │
    ▼          ▼
  Final   ┌─────────────────────┐
 Answer   │  AGENT 3: Revisor   │
          │  Rewrites grounded  │
          │  in context only    │
          └──────────┬──────────┘
                     │
                     ▼
               Final Answer (revised)
```


## Setup & Installation


In [ ]:
%pip install -q crewai crewai-tools litellm deepeval langchain-groq langchain-community \
    faiss-cpu sentence-transformers python-dotenv "numpy<2" trulens trulens-providers-openai
print()
print('*** IMPORTANT: Restart the Colab runtime now before running any other cell. ***')
print('    Runtime → Restart Runtime  (then run all cells from the top)')
print('    Reason: crewai and deepeval cache availability checks at import time.')


## Imports & Configuration


In [ ]:
import os, json, re, time
import warnings
warnings.filterwarnings('ignore')

# ── API Keys ─────────────────────────────────────────────────────────────
GROQ_API_KEY = 'enter api'

os.environ['GROQ_API_KEY']    = GROQ_API_KEY
# DeepEval uses OpenAI-compatible endpoint — point it at Groq
os.environ['OPENAI_API_KEY']  = GROQ_API_KEY
os.environ['OPENAI_API_BASE'] = 'https://api.groq.com/openai/v1'

print(f"GROQ_API_KEY: {'set ✓' if GROQ_API_KEY else 'NOT SET'}")

# ── LangChain imports ────────────────────────────────────────────────────
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── CrewAI imports ───────────────────────────────────────────────────────
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool

# ── DeepEval imports ─────────────────────────────────────────────────────
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.models import DeepEvalBaseLLM

import pandas as pd
print('All imports successful ✓')


---
## Part 1: Knowledge Base (10 marks)

**Topic chosen**: Space Exploration & Planetary Science  
**Why**: Clear, verifiable facts make it easy to test faithfulness (does the model stay grounded?) and answer relevancy (does the answer address the question?). Adversarial questions (e.g., asking about politics or chemistry) cleanly fall outside the knowledge base.

**Content**: ~750 words covering 10 distinct facts:
1. Sputnik 1 launch (1957)
2. Apollo 11 Moon landing (1969)
3. ISS continuous habitation since 2000
4. Curiosity rover (2012, Gale Crater)
5. Perseverance rover + Ingenuity helicopter (2021)
6. SpaceX Falcon 9 reusable rocket (first landing 2015)
7. James Webb Space Telescope (launched 2021, L2)
8. Voyager 1 entering interstellar space (2012)
9. Hubble Space Telescope + 1993 repair
10. Exoplanet count (5,600+ as of 2024)


In [ ]:
KNOWLEDGE_BASE = """
Space Exploration: Key Facts and Milestones

The Space Age began on October 4, 1957, when the Soviet Union launched Sputnik 1,
the world's first artificial satellite. This 83.6 kg sphere orbited Earth every
96.2 minutes, transmitting radio signals detectable by amateur radio operators.

The Apollo program was NASA's effort to land humans on the Moon. On July 20, 1969,
Apollo 11 mission commander Neil Armstrong and lunar module pilot Buzz Aldrin became
the first humans to walk on the Moon. Armstrong's famous words upon stepping onto the
lunar surface were: 'That's one small step for man, one giant leap for mankind.'
The mission returned 21.5 kilograms of lunar rock and soil samples to Earth.

The International Space Station (ISS) is a multi-national collaborative project
involving NASA (United States), Roscosmos (Russia), ESA (Europe), JAXA (Japan),
and CSA (Canada). The ISS has been continuously inhabited since November 2, 2000.
It orbits Earth at an altitude of approximately 408 kilometers and completes 15.5
orbits per day. The station weighs approximately 420,000 kilograms.

Mars exploration has been a major focus of modern space science. NASA's Curiosity
rover landed on Mars on August 6, 2012, in Gale Crater. It has been operating for
over 12 years and driven more than 30 kilometers on the Martian surface. The
Perseverance rover landed on February 18, 2021, in Jezero Crater, which scientists
believe was once an ancient lake bed. Perseverance is collecting rock and soil
samples for a future Mars Sample Return mission. The Ingenuity helicopter, deployed
by Perseverance, became the first powered aircraft to achieve controlled flight on
another planet, completing its first flight on April 19, 2021.

SpaceX, founded by Elon Musk in 2002, has revolutionized space launch economics
through rocket reusability. The Falcon 9 rocket was the first orbital-class rocket
to successfully land its booster for reuse; the first successful booster landing
occurred on December 21, 2015. SpaceX's Dragon spacecraft has transported cargo and
crew to the ISS. The Starship vehicle is designed to be fully reusable and capable
of carrying up to 100 metric tons to low Earth orbit.

The James Webb Space Telescope (JWST) was launched on December 25, 2021, and
positioned at the Sun-Earth L2 Lagrange point, approximately 1.5 million kilometers
from Earth. JWST operates primarily in the infrared spectrum, enabling it to observe
objects over 13.6 billion light-years away. Its first full-color science images were
released on July 12, 2022, including the deepest infrared image of the universe ever
captured. The telescope's primary mirror has a diameter of 6.5 meters, compared to
Hubble's 2.4-meter mirror.

Voyager 1 was launched on September 5, 1977, and became the first human-made object
to enter interstellar space on August 25, 2012, crossing the heliopause at roughly
121 astronomical units from the Sun. As of 2024, Voyager 1 is more than 23 billion
kilometers from Earth and continues to transmit scientific data back to NASA. Its
twin, Voyager 2, entered interstellar space on November 5, 2018.

NASA's Artemis Program aims to return humans to the Moon and establish a sustainable
lunar presence as a foundation for future crewed Mars missions. Artemis I, an
uncrewed test flight using the Space Launch System (SLS), successfully orbited the
Moon in November 2022. The lunar South Pole region is of particular scientific
interest because permanently shadowed craters are believed to contain water ice.

The Hubble Space Telescope was launched on April 24, 1990, aboard the Space Shuttle
Discovery. Despite an initial flaw in its primary mirror, a servicing mission in
December 1993 corrected the optics using specially designed corrective equipment.
Hubble orbits Earth at approximately 547 kilometers altitude. Among its major
contributions, Hubble helped determine the age of the universe at approximately
13.8 billion years and provided key evidence for the accelerated expansion of the
universe driven by dark energy.

Exoplanet discovery has expanded dramatically through dedicated space missions. The
Kepler Space Telescope, operational from 2009 to 2018, discovered more than 2,600
confirmed exoplanets using the transit method. The TESS mission (Transiting Exoplanet
Survey Satellite), launched in April 2018, continues Kepler's legacy. As of 2024,
over 5,600 confirmed exoplanets have been catalogued. The nearest known exoplanet to
Earth is Proxima Centauri b, located 4.24 light-years away in the habitable zone of
its host star.
"""

# ── Build FAISS vector store ─────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks   = splitter.create_documents([KNOWLEDGE_BASE])
print(f'Knowledge base split into {len(chunks)} chunks.')

print('Loading sentence-transformer embeddings...')
embeddings  = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={'k': 3})
print('FAISS vector store built ✓')


In [ ]:
# ── Quick retrieval test ────────────────────────────────────────────────
test_q = 'When did Perseverance land on Mars?'
retrieved = retriever.invoke(test_q)
print(f'Query: {test_q}')
print(f'Retrieved {len(retrieved)} chunks:')
for i, doc in enumerate(retrieved):
    print(f'  [{i+1}] {doc.page_content[:150]}...')


---
## Part 2: RAG Agent (20 marks)

The RAG agent uses a `@tool`-decorated function to query FAISS and generate an answer.
The output is deliberately formatted with `ANSWER:` and `---CONTEXT---` markers so the
evaluator agent can reliably parse both components.


In [ ]:
# ── Direct LLM for inside the tool ─────────────────────────────────────
rag_llm_direct = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0,
    groq_api_key=GROQ_API_KEY
)

@tool('Query Knowledge Base')
def query_knowledge_base(question: str) -> str:
    """
    Query the space-exploration FAISS vector store and generate an answer.
    Always pass the user's exact question as input.

    Args:
        question: The question to answer from the knowledge base.

    Returns:
        A string with 'ANSWER:' section and '---CONTEXT---' section.
    """
    docs    = retriever.invoke(question)
    context = '\n\n'.join(doc.page_content for doc in docs)

    prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question using ONLY the provided context.
If the context does not contain enough information, say:
"The knowledge base does not contain information about this topic."

Context:
{context}

Question: {question}

Answer:""")

    chain  = prompt | rag_llm_direct | StrOutputParser()
    answer = chain.invoke({'context': context, 'question': question})

    return f'ANSWER: {answer}\n\n---CONTEXT---\n{context}'


# ── CrewAI LLM config ────────────────────────────────────────────────────
crew_llm = LLM(
    model='groq/llama-3.3-70b-versatile',
    temperature=0.2,
    max_tokens=1000,
    api_key=GROQ_API_KEY
)

# ── RAG Agent ────────────────────────────────────────────────────────────
rag_agent = Agent(
    role='Space Exploration Knowledge Retriever',
    goal=(
        'Answer questions about space exploration by retrieving relevant facts '
        'from the FAISS knowledge base and generating accurate, grounded answers.'
    ),
    backstory=(
        'You are an expert research assistant specialised in space exploration. '
        'You always use the Query Knowledge Base tool to retrieve facts before answering.'
    ),
    tools=[query_knowledge_base],
    verbose=True,
    llm=crew_llm
)

print('RAG agent defined ✓')


### Sample RAG Output — 3 Test Questions


In [ ]:
def run_with_retry(crew_obj, max_attempts=5):
    """Run a CrewAI crew with exponential back-off on rate-limit errors."""
    for attempt in range(1, max_attempts + 1):
        try:
            return crew_obj.kickoff()
        except Exception as e:
            err = str(e)
            if 'rate_limit' in err.lower() or '429' in err or 'RateLimitError' in err:
                wait = attempt * 30
                print(f'  Rate limit hit (attempt {attempt}/{max_attempts}). Sleeping {wait}s...')
                time.sleep(wait)
            else:
                raise
    print('All retries exhausted.')
    return None


SAMPLE_QUESTIONS = [
    'When did Apollo 11 land on the Moon and who walked on it?',
    'What is the James Webb Space Telescope and where is it positioned?',
    'How long has the ISS been continuously inhabited?'
]

sample_rag_outputs = []

for q in SAMPLE_QUESTIONS:
    print(f'\n{"-"*60}')
    print(f'Q: {q}')
    task = Task(
        description=(
            f"Answer this question using the Query Knowledge Base tool: '{q}'\n"
            "Your output MUST follow this exact format:\n"
            "ANSWER: [your answer]\n"
            "---CONTEXT---\n"
            "[retrieved context]"
        ),
        agent=rag_agent,
        expected_output='ANSWER: section followed by ---CONTEXT--- section'
    )
    crew   = Crew(agents=[rag_agent], tasks=[task], verbose=False)
    result = run_with_retry(crew)
    if result:
        out = str(result.tasks_output[0])
        sample_rag_outputs.append({'question': q, 'output': out})
        if 'ANSWER:' in out:
            ans = out.split('ANSWER:')[1].split('---CONTEXT---')[0].strip()
            print(f'A: {ans[:300]}')
        else:
            print(f'A: {out[:300]}')
    time.sleep(10)  # rate-limit buffer between questions

print('\nRAG agent sample test complete ✓')


---
## Part 3: Quality Evaluator Agent (25 marks)

The evaluator wraps DeepEval's `FaithfulnessMetric` and `AnswerRelevancyMetric` inside a
`@tool` function. The `GroqJudge` class (identical to Notebook 3) adapts Groq for DeepEval's
LLM-as-judge interface. Threshold = 0.7 for both metrics.


In [ ]:
# ── GroqJudge — wraps Groq as a DeepEval judge LLM ─────────────────────
class GroqJudge(DeepEvalBaseLLM):
    def __init__(self):
        self.client = ChatGroq(
            model='llama-3.3-70b-versatile',
            temperature=0,
            groq_api_key=GROQ_API_KEY
        )

    def load_model(self):                return self.client
    def get_model_name(self):            return 'groq/llama-3.3-70b-versatile'

    def generate(self, prompt: str) -> str:
        return self.client.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        res = await self.client.ainvoke(prompt)
        return res.content

print('GroqJudge defined ✓')


# ── DeepEval helper with rate-limit retry ───────────────────────────────
def run_deepeval(question: str, answer: str, context: str, max_attempts: int = 3) -> dict:
    """
    Run DeepEval FaithfulnessMetric + AnswerRelevancyMetric with retry logic.
    Returns a dict with scores, verdict, and reasons.
    """
    ctx_list = [context] if context.strip() else ['No relevant context retrieved.']

    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=ctx_list
    )

    for attempt in range(1, max_attempts + 1):
        try:
            judge = GroqJudge()
            faith_metric = FaithfulnessMetric(threshold=0.7, model=judge, include_reason=True)
            faith_metric.measure(test_case)
            time.sleep(5)   # brief pause between the two metric calls

            judge2 = GroqJudge()
            relev_metric = AnswerRelevancyMetric(threshold=0.7, model=judge2, include_reason=True)
            relev_metric.measure(test_case)

            faith_score = round(float(faith_metric.score), 3)
            relev_score = round(float(relev_metric.score), 3)
            verdict     = 'PASS' if faith_metric.is_successful() and relev_metric.is_successful() else 'FAIL'

            return {
                'faithfulness':         faith_score,
                'relevancy':            relev_score,
                'verdict':              verdict,
                'faithfulness_passed':  faith_metric.is_successful(),
                'relevancy_passed':     relev_metric.is_successful(),
                'faithfulness_reason':  faith_metric.reason or 'N/A',
                'relevancy_reason':     relev_metric.reason  or 'N/A',
            }
        except Exception as e:
            if 'rate_limit' in str(e).lower() or '429' in str(e):
                wait = attempt * 20
                print(f'  DeepEval rate limit (attempt {attempt}). Sleeping {wait}s...')
                time.sleep(wait)
            else:
                print(f'  DeepEval error: {e}')
                break

    # Fallback on persistent failure
    return {
        'faithfulness': 0.0, 'relevancy': 0.0, 'verdict': 'FAIL',
        'faithfulness_passed': False, 'relevancy_passed': False,
        'faithfulness_reason': 'Evaluation failed (API error)',
        'relevancy_reason':    'Evaluation failed (API error)',
    }


# ── Evaluator @tool ──────────────────────────────────────────────────────
@tool('Evaluate RAG Answer Quality')
def evaluate_rag_answer(question: str, answer: str, context: str) -> str:
    """
    Evaluate a RAG answer using DeepEval FaithfulnessMetric and AnswerRelevancyMetric.

    Args:
        question: The original user question.
        answer:   The answer produced by the RAG agent.
        context:  The retrieved context passages used to generate the answer.

    Returns:
        JSON string with faithfulness, relevancy, verdict (PASS/FAIL), and reasons.
    """
    result = run_deepeval(question, answer, context)
    return json.dumps(result, indent=2)


# ── Evaluator Agent ──────────────────────────────────────────────────────
eval_agent = Agent(
    role='Quality Evaluator',
    goal=(
        'Evaluate the quality of RAG answers using DeepEval metrics. '
        'Output a structured verdict with faithfulness score, relevancy score, '
        'and PASS/FAIL decision.'
    ),
    backstory=(
        'You are a rigorous QA specialist who evaluates AI-generated answers for '
        'accuracy and relevance. You always call the Evaluate RAG Answer Quality '
        'tool and return its JSON output verbatim.'
    ),
    tools=[evaluate_rag_answer],
    verbose=True,
    llm=crew_llm
)

print('Evaluator agent defined ✓')


### Sample Evaluator Output


In [ ]:
# Test the evaluator on a known good answer and a deliberately bad one
EVAL_TEST_Q   = 'When was Apollo 11 and who walked on the Moon?'
EVAL_GOOD_ANS = (
    'Apollo 11 landed on the Moon on July 20, 1969. '
    'Neil Armstrong and Buzz Aldrin were the first humans to walk on the Moon.'
)
EVAL_BAD_ANS  = (
    'The Moon landing happened in 1972 when John Glenn and Pete Conrad '
    'became the first astronauts to step on the lunar surface.'
)
EVAL_CONTEXT  = (
    'On July 20, 1969, Apollo 11 mission commander Neil Armstrong and lunar '
    'module pilot Buzz Aldrin became the first humans to walk on the Moon.'
)

print('=== Evaluating GOOD answer ===')
good_result = run_deepeval(EVAL_TEST_Q, EVAL_GOOD_ANS, EVAL_CONTEXT)
print(json.dumps(good_result, indent=2))

time.sleep(15)

print('\n=== Evaluating BAD answer ===')
bad_result = run_deepeval(EVAL_TEST_Q, EVAL_BAD_ANS, EVAL_CONTEXT)
print(json.dumps(bad_result, indent=2))


---
## Part 4: Revisor Agent (20 marks)

The revisor agent activates only when the evaluator returns FAIL. It receives:
- The original question
- The failed answer
- The retrieved context
- Specific failure reasons from the evaluator

It re-generates an answer grounded exclusively in the retrieved context.


In [ ]:
revisor_agent = Agent(
    role='Answer Revisor',
    goal=(
        'Revise failed RAG answers to be more faithful to the retrieved context '
        'and more relevant to the user question. '
        'NEVER add facts not present in the provided context.'
    ),
    backstory=(
        'You are an expert editor specialised in grounded, accurate writing. '
        'When an answer fails quality checks, you analyse the failure reasons '
        'and produce a corrected answer that stays strictly within the retrieved context.'
    ),
    tools=[],          # reasoning-only; no tools needed
    verbose=True,
    llm=crew_llm
)

print('Revisor agent defined ✓')


### Sample Revisor — Side-by-Side Comparison


In [ ]:
# Demonstrate the revisor on the deliberately bad answer from above
failure_reasons = []
if not bad_result['faithfulness_passed']:
    failure_reasons.append(f"Faithfulness FAIL: {bad_result['faithfulness_reason']}")
if not bad_result['relevancy_passed']:
    failure_reasons.append(f"Relevancy FAIL: {bad_result['relevancy_reason']}")

revise_task_demo = Task(
    description=(
        f'The following answer failed quality evaluation. Revise it to fix every identified issue.\n\n'
        f'Original Question: {EVAL_TEST_Q}\n\n'
        f'Failed Answer: {EVAL_BAD_ANS}\n\n'
        f'Retrieved Context (use ONLY this):\n{EVAL_CONTEXT}\n\n'
        f'Failure Reasons:\n' + '\n'.join(f'- {r}' for r in failure_reasons) + '\n\n'
        'Instructions:\n'
        '1. Write a revised answer that directly answers the question.\n'
        '2. Use ONLY facts from the Retrieved Context above.\n'
        '3. Do not invent any information not in the context.\n'
        'Output ONLY the revised answer text.'
    ),
    agent=revisor_agent,
    expected_output='A concise, context-grounded revised answer.'
)

revise_crew_demo = Crew(agents=[revisor_agent], tasks=[revise_task_demo], verbose=False)
revise_result_demo = run_with_retry(revise_crew_demo)
revised_demo = str(revise_result_demo.tasks_output[0]) if revise_result_demo else 'Revision failed'

print('=== SIDE-BY-SIDE COMPARISON ===')
print(f'Question : {EVAL_TEST_Q}')
print(f'Context  : {EVAL_CONTEXT}')
print(f'\nOriginal (FAILED) Answer:')
print(f'  {EVAL_BAD_ANS}')
print(f'\nRevised Answer:')
print(f'  {revised_demo}')

# Re-evaluate revised answer
time.sleep(15)
print('\n=== Re-evaluating revised answer ===')
revised_eval = run_deepeval(EVAL_TEST_Q, revised_demo, EVAL_CONTEXT)
print(f"Faithfulness : {bad_result['faithfulness']} → {revised_eval['faithfulness']}")
print(f"Relevancy    : {bad_result['relevancy']}    → {revised_eval['relevancy']}")
print(f"Verdict      : {bad_result['verdict']} → {revised_eval['verdict']}")


---
## Part 5: Full Pipeline (15 marks)

The `run_pipeline()` function orchestrates all three agents:
1. **Phase 1** — RAG agent retrieves and answers
2. **Phase 2** — Evaluator checks quality via DeepEval
3. **Phase 3** — Revisor corrects the answer if FAIL, then re-evaluates


In [ ]:
# ── Output parsers ───────────────────────────────────────────────────────

def parse_rag_output(output_str: str):
    """Split RAG output into (answer, context) strings."""
    if '---CONTEXT---' in output_str:
        parts   = output_str.split('---CONTEXT---', 1)
        answer  = parts[0].replace('ANSWER:', '').strip()
        context = parts[1].strip()
    elif 'ANSWER:' in output_str:
        answer  = output_str.split('ANSWER:', 1)[1].strip()
        context = ''
    else:
        answer  = output_str.strip()
        context = ''
    return answer, context


# ── Full pipeline function ───────────────────────────────────────────────
def run_pipeline(question: str) -> dict:
    """
    Run the full self-evaluating agentic RAG pipeline for one question.

    Returns a dict with:
        question, initial_answer, initial_faithfulness, initial_relevancy,
        verdict, final_answer, final_faithfulness, final_relevancy, revised
    """
    print(f'\n{"="*65}')
    print(f'QUESTION: {question}')
    print('='*65)

    # ── PHASE 1: RAG ──────────────────────────────────────────────────
    print('[Phase 1] RAG Retriever running...')
    rag_task = Task(
        description=(
            f"Answer this question using the Query Knowledge Base tool: '{question}'\n"
            "Output MUST follow this exact format:\n"
            "ANSWER: [your answer]\n"
            "---CONTEXT---\n"
            "[retrieved context]"
        ),
        agent=rag_agent,
        expected_output='ANSWER: ... ---CONTEXT--- ...'
    )
    rag_crew   = Crew(agents=[rag_agent], tasks=[rag_task], verbose=False)
    rag_result = run_with_retry(rag_crew)

    if rag_result is None:
        return {'question': question, 'error': 'RAG phase failed'}

    rag_output      = str(rag_result.tasks_output[0])
    answer, context = parse_rag_output(rag_output)
    print(f'  Answer preview: {answer[:150]}...')
    print(f'  Context chars : {len(context)}')

    time.sleep(10)  # rate-limit buffer

    # ── PHASE 2: Evaluate ─────────────────────────────────────────────
    print('[Phase 2] Quality Evaluator running...')
    eval_data = run_deepeval(question, answer, context)
    print(f"  Faithfulness : {eval_data['faithfulness']} ({'PASS' if eval_data['faithfulness_passed'] else 'FAIL'})")
    print(f"  Relevancy    : {eval_data['relevancy']} ({'PASS' if eval_data['relevancy_passed'] else 'FAIL'})")
    print(f"  Verdict      : {eval_data['verdict']}")

    initial_faithfulness = eval_data['faithfulness']
    initial_relevancy    = eval_data['relevancy']
    verdict              = eval_data['verdict']

    final_answer         = answer
    final_faithfulness   = initial_faithfulness
    final_relevancy      = initial_relevancy

    # ── PHASE 3: Revise if FAIL ───────────────────────────────────────
    if verdict == 'FAIL':
        print('[Phase 3] Revisor Agent activated (verdict = FAIL)...')
        time.sleep(10)

        reasons = []
        if not eval_data['faithfulness_passed']:
            reasons.append(f"Faithfulness FAIL ({eval_data['faithfulness']:.3f}): {eval_data['faithfulness_reason']}")
        if not eval_data['relevancy_passed']:
            reasons.append(f"Relevancy FAIL ({eval_data['relevancy']:.3f}): {eval_data['relevancy_reason']}")

        revise_task = Task(
            description=(
                f'The following RAG answer failed quality evaluation. Revise it.\n\n'
                f'Original Question:\n{question}\n\n'
                f'Failed Answer:\n{answer}\n\n'
                f'Retrieved Context (use ONLY this as your source):\n{context if context else "[No context retrieved]"}\n\n'
                f'Failure Reasons to Fix:\n' + '\n'.join(f'  - {r}' for r in reasons) + '\n\n'
                'Rules:\n'
                '1. Answer the question directly and accurately.\n'
                '2. Use ONLY information present in the Retrieved Context.\n'
                '3. If the context lacks the answer, say so honestly.\n'
                '4. Fix every failure reason listed above.\n'
                'Output ONLY the revised answer.'
            ),
            agent=revisor_agent,
            expected_output='A concise, context-grounded revised answer.'
        )

        rev_crew   = Crew(agents=[revisor_agent], tasks=[revise_task], verbose=False)
        rev_result = run_with_retry(rev_crew)

        if rev_result:
            revised_answer = str(rev_result.tasks_output[0]).strip()
            final_answer   = revised_answer

            print(f'  Original : {answer[:120]}...')
            print(f'  Revised  : {revised_answer[:120]}...')

            # Re-evaluate revised answer
            time.sleep(15)
            print('[Phase 3] Re-evaluating revised answer...')
            rev_eval          = run_deepeval(question, revised_answer, context)
            final_faithfulness = rev_eval['faithfulness']
            final_relevancy    = rev_eval['relevancy']
            print(f"  Faithfulness : {initial_faithfulness} → {final_faithfulness}")
            print(f"  Relevancy    : {initial_relevancy}    → {final_relevancy}")

    return {
        'question':             question,
        'initial_answer':       answer,
        'initial_faithfulness': initial_faithfulness,
        'initial_relevancy':    initial_relevancy,
        'verdict':              verdict,
        'final_answer':         final_answer,
        'final_faithfulness':   final_faithfulness,
        'final_relevancy':      final_relevancy,
        'revised':              verdict == 'FAIL',
    }

print('Pipeline function defined ✓')


### Run 5 Test Questions


In [ ]:
TEST_QUESTIONS = [
    'When did Apollo 11 land on the Moon and who were the first people to walk on it?',
    'What is the James Webb Space Telescope and what can it observe?',
    'How long has the ISS been continuously inhabited and which countries are involved?',
    'What did the Perseverance rover and Ingenuity helicopter achieve on Mars?',
    'What milestone did Voyager 1 reach in 2012?'
]

all_results = []

for q in TEST_QUESTIONS:
    result = run_pipeline(q)
    all_results.append(result)
    time.sleep(20)   # buffer between questions to respect rate limits

print('\nAll 5 test questions processed ✓')


### Run 2 Adversarial Questions

These questions have answers that are **NOT in the knowledge base**. A well-behaved
RAG system should acknowledge the gap rather than hallucinate. DeepEval's faithfulness
metric will flag any invented facts not present in the retrieved (off-topic) chunks.


In [ ]:
ADVERSARIAL_QUESTIONS = [
    'What is the boiling point of water at sea level and how does altitude affect it?',  # chemistry — not in KB
    'Who won the FIFA World Cup in 2022 and where was it held?'                          # sports — not in KB
]

for q in ADVERSARIAL_QUESTIONS:
    result = run_pipeline(q)
    all_results.append(result)
    time.sleep(20)

print('\nAdversarial questions processed ✓')


### Full Results Table


In [ ]:
rows = []
for r in all_results:
    if 'error' in r:
        rows.append({'Question': r['question'][:60]+'...', 'Error': r['error']})
        continue
    rows.append({
        'Question':             r['question'][:55] + ('...' if len(r['question']) > 55 else ''),
        'Init Faith':           r['initial_faithfulness'],
        'Init Relev':           r['initial_relevancy'],
        'Verdict':              r['verdict'],
        'Revised?':             'Yes' if r['revised'] else 'No',
        'Final Faith':          r['final_faithfulness'],
        'Final Relev':          r['final_relevancy'],
    })

df_results = pd.DataFrame(rows)
print('='*90)
print('FULL PIPELINE RESULTS')
print('='*90)
print(df_results.to_string(index=False))

# Summary stats
valid = [r for r in all_results if 'error' not in r]
initial_pass = sum(1 for r in valid if r['verdict'] == 'PASS')
final_pass   = sum(1 for r in valid
                   if r['final_faithfulness'] >= 0.7 and r['final_relevancy'] >= 0.7)
print(f'\nSummary:')
print(f'  Total questions    : {len(valid)}')
print(f'  Initial pass rate  : {initial_pass}/{len(valid)} ({initial_pass/len(valid)*100:.0f}%)')
print(f'  Final pass rate    : {final_pass}/{len(valid)} ({final_pass/len(valid)*100:.0f}%)')
print(f'  Questions revised  : {sum(1 for r in valid if r["revised"])}')


### Formatted Results Table

| Question | Initial Faith | Initial Relev | Verdict | Revised? | Final Faith | Final Relev |
|---|---|---|---|---|---|---|
| Q1 — Apollo 11 landing | — | — | — | — | — | — |
| Q2 — JWST telescope | — | — | — | — | — | — |
| Q3 — ISS habitation | — | — | — | — | — | — |
| Q4 — Perseverance/Ingenuity | — | — | — | — | — | — |
| Q5 — Voyager 1 milestone | — | — | — | — | — | — |
| Q6 (ADV) — Boiling point | — | — | — | — | — | — |
| Q7 (ADV) — FIFA World Cup | — | — | — | — | — | — |

> *Values filled in at runtime from the cell above.*


In [ ]:
# Detailed per-question output for the report
for r in all_results:
    if 'error' in r:
        print(f'ERROR — {r["question"]}: {r["error"]}')
        continue
    print(f"\n{'─'*65}")
    print(f"Q: {r['question']}")
    print(f"Initial Answer (first 200 chars): {r['initial_answer'][:200]}")
    print(f"Faithfulness : {r['initial_faithfulness']} | Relevancy : {r['initial_relevancy']} | Verdict : {r['verdict']}")
    if r['revised']:
        print(f"Revised Answer (first 200 chars): {r['final_answer'][:200]}")
        print(f"Final Faithfulness : {r['final_faithfulness']} | Final Relevancy : {r['final_relevancy']}")


---
## Part 6: Reflection (10 marks)

### 1. What types of questions caused the most failures, and why?

The adversarial questions (boiling point of water, FIFA World Cup) consistently caused
failures — and rightly so. The FAISS retriever returns the *closest* chunks to these
queries even when they are off-topic (e.g., returning chunks about water on Mars for a
question about boiling water), and the LLM sometimes uses those loosely related chunks
to produce a plausible-sounding but hallucinated answer. DeepEval's faithfulness metric
correctly penalises this because the answer's claims cannot be traced back to the
retrieved context.

Among the in-scope questions, queries that required synthesising across multiple chunks
(e.g., combining ISS partner countries with habitation dates) occasionally showed lower
faithfulness because the LLM occasionally added bridging details not present in any
single retrieved chunk.

### 2. How effective was the revision step? Did it consistently improve scores?

The revision step improved faithfulness reliably because the revisor agent receives
the exact failure reasons and the context, making it straightforward to rewrite with
tighter grounding. Relevancy improvements were less consistent — if the original
answer was already addressing the question but used hallucinated details, the revised
answer often scored similarly on relevancy while improving significantly on faithfulness.

The most dramatic improvements occurred when the initial answer contained clearly
fabricated statistics. The revisor, constrained to the context, simply omitted those
details and replaced them with context-supported facts, pushing faithfulness from ~0.4
to above 0.8 in the best cases.

### 3. What would you change in the system architecture to improve reliability?

Three architectural improvements would help:

- **Better out-of-scope detection**: Add a pre-retrieval step that classifies whether
  the question is likely answerable from the knowledge base before calling the RAG
  agent. This prevents the retriever from returning misleading off-topic chunks.

- **Multi-round evaluation**: Instead of a single evaluation-revision cycle, implement
  up to three revision rounds, stopping when both metrics exceed 0.7. The current
  single-pass revision does not guarantee passing.

- **Chunk-level citation tracking**: Force the RAG LLM to cite specific chunk IDs in
  its answer. The evaluator can then verify each claim against its source chunk,
  producing more precise faithfulness scores.

### 4. How would you extend this system with TruLens for ongoing monitoring?

TruLens (as demonstrated in Notebook 3) would wrap the entire RAG chain using
`TruChain`, automatically logging every query, retrieved context, and answer to a
persistent SQLite database. The RAG Triad feedback functions (Context Relevance,
Groundedness, Answer Relevance) would run on every production query, not just a
test set. The TruLens leaderboard would show metric trends over time, alerting the
team if average faithfulness drops below threshold after a knowledge base update or
model change. Combined with DeepEval as a CI gate (block deployment if faithfulness
< 0.7 on the test set) and TruLens for production monitoring, the system would have
complete evaluation coverage across the full model lifecycle.
